# McGill COMP551, MINI-PROJECT 3: Odd-One-Out Image Groups

**Kaggle team name**

## Overview

5 grayscale images per group; 4 share a property, 1 is the outlier. Task: predict outlier index (0–4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.load("datasets/x_train.npy")
y = np.load("datasets/y_train.npy")
print(f"Train: {x.shape[0]} groups, 5 images each, {x.shape[2]}x{x.shape[3]}")

## Step 1 — Data & Preprocessing

Preprocessing: crop to object, center in 32×32, extract shape metadata (Hu moments, contours, PCA).

In [ ]:
!python -m src.preprocess --data-dir datasets

## Step 2 — Model

**OddOneOutNet:** TinyCNN + meta MLP → set attention + archetype clustering + pairwise relations → 5-way classifier.

**Params:** < 25,000.

In [ ]:
import json
import numpy as np
import torch
from src.nets import OddOneOutNet, load_npz_arrays

z = np.load("datasets/processed_train_best.npz", allow_pickle=True)
meta_dim = z["meta"].shape[-1]
model = OddOneOutNet(meta_dim=meta_dim)
model.load_state_dict(torch.load("best_model.pt", map_location="cpu"))

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")
assert total_params <= 25_000, f"Model too large: {total_params:,} > 25,000"

## Step 3 — Training (optional)

Run to retrain. Otherwise we use the saved `best_model.pt`.

In [ ]:
# Uncomment to retrain:
# !python -m src.nets --data-dir datasets

## Step 4 — Public Test Accuracy

In [ ]:
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader
from src.nets import DictDataset, predict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

test_arrays = load_npz_arrays("datasets/processed_test_best.npz")
y_test = np.load("datasets/y_test.npy")
n = len(y_test)
pub_arrays = {k: v[:n] for k, v in test_arrays.items() if isinstance(v, np.ndarray)}
pub_ds = DictDataset(pub_arrays, y=y_test)
pub_loader = DataLoader(pub_ds, batch_size=64, shuffle=False)

pub_preds = predict(model, pub_loader, device)
acc = accuracy_score(y_test, pub_preds)
print(f"Public test accuracy (must match Kaggle public leaderboard): {acc * 100:.2f}%")

## Step 5 — Kaggle Submission CSV

In [ ]:
from src.nets import generate_csv_kaggle

test_ds = DictDataset(test_arrays, y=None)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
yh_test = predict(model, test_loader, device)

generate_csv_kaggle(yh_test)
print(f"Saved {len(yh_test)} predictions to predicted_labels.csv")